# Data Exploration and Anomaly Detection

This notebook performs comprehensive data exploration, including time-series visualization and anomaly detection for the Enedis energy consumption dataset.

In [1]:
import polars as pl
import polars.selectors as cs
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
from scipy import stats

from src.config import PROCESSED_DATA_DIR, FIGURES_DIR
from utils.plots import plot_time_serie, plot_time_serie_monthly, plot_distribution, detect_outlier, plot_acf_decomposition, plot_pacf_decomposition
from utils.stats import test_stationarity

# Load configurations
import notebooks.config.polars_config
import notebooks.config.matplotlib_config
import notebooks.config.seaborn_config
import notebooks.config.statsmodels_config

2026-04-21 05:01:20.930 | INFO     | src.config:<module>:11 - PROJ_ROOT path is: /home/abdelaziz/Desktop/enedis_pipeline/data_analytics


## Configuration and Helper Functions

In [2]:
# Years to explore
YEARS = range(2021, 2027)

def get_save_path(year, category, filename):
    """Generates and ensures the existence of the save directory for figures."""
    path = FIGURES_DIR / str(year) / category / f"{filename}.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    return path

def process_year_exploration(year):
    """Performs exploration for a specific year's dataset."""
    data_path = PROCESSED_DATA_DIR / f"enedis_dataset_year_{year}.parquet"
    if not data_path.exists():
        print(f"Dataset for year {year} not found at {data_path}. Skipping.")
        return None
    
    df = pl.read_parquet(data_path)
    print(f"Loaded dataset for year {year} with {df.height} rows.")
    return df

## Anomaly Detection and Labeling

We detect anomalies for `consommation`, `production`, `injection`, and `soutirage` columns. 
For columns matching the pattern `consommation_telerelevee`, we add a boolean indicator to the dataset.

In [3]:
def label_and_plot_anomalies(df, year):
    """
    Detects and plots anomalies for specific categories.
    Labels 'consommation_telerelevee' columns with anomaly status.
    """
    # 1. Label columns with 'consommation_telerelevee' pattern
    telerelvee_cols = [col for col in df.columns if "consommation_telerelevee" in col]
    for col in telerelvee_cols:
        anomaly_col_name = f"is_anomaly_{col}"
        df = df.with_columns(
            detect_outlier(df, col).alias(anomaly_col_name)
        )
    
    # 2. Plot anomalies for Consommation, Production, Injection, Soutirage only
    anomaly_categories = {
        "Consommation": [col for col in df.columns if "consommation" in col and not col.startswith("is_anomaly_")],
        "Production": [col for col in df.columns if "production" in col and not col.startswith("is_anomaly_")],
        "Injection": [col for col in df.columns if "injection" in col and not col.startswith("is_anomaly_")],
        "Soutirage": [col for col in df.columns if "soutirage" in col and not col.startswith("is_anomaly_")]
    }
    
    for category, columns in anomaly_categories.items():
        for col in columns:
            if col in df.columns:
                save_path = get_save_path(year, category, f"Anomaly_{col}")
                plot_distribution(df, col, "timestamp", save_path=save_path)
                
    return df

## Standard Time Series Exploration

Refactored plotting calls for each category, following the original exploration logic.

In [4]:
def run_standard_exploration(df, year):
    """Runs standard time series plots (Yearly and Monthly) for all categories."""
    
    # Consommation
    save_path = get_save_path(year, "Consommation", "yearly").parent
    plot_time_serie(df, "timestamp", f"Consommation-{year}", save_path, "consommation_telerelevee", "consommation_profilee")
    plot_time_serie_monthly(df, "timestamp", f"Consommation-{year}", save_path, "consommation_telerelevee", "consommation_profilee")
    
    # Consommation HTA
    save_path = get_save_path(year, "Consommation_HTA", "yearly").parent
    plot_time_serie(df, "timestamp", f"Consommation_HTA-{year}", save_path, "consommation_telerelevee_hta", "consommation_profilee_ent_hta")
    plot_time_serie_monthly(df, "timestamp", f"Consommation_HTA_Monthly-{year}", save_path, "consommation_telerelevee_hta", "consommation_profilee_ent_hta")

    # Consommation BTSUP
    save_path = get_save_path(year, "Consommation_BTSUP", "yearly").parent
    plot_time_serie(df, "timestamp", f"Consommation_BTSUP-{year}", save_path, "consommation_telerelevee_btsup", "consommation_profilee_ent_bt")
    plot_time_serie_monthly(df, "timestamp", f"Consommation_BTSUP-{year}", save_path, "consommation_telerelevee_btsup", "consommation_profilee_ent_bt")

    # Consommation Professionnelle
    save_path = get_save_path(year, "Consommation_PRO", "yearly").parent
    plot_time_serie(df, "timestamp", f"Consommation_PRO_Monthly-{year}", save_path, "consommation_telerelevee_professionnelle", "consommation_profilee_pro")
    plot_time_serie_monthly(df, "timestamp", f"Consommation_PRO-{year}", save_path, "consommation_telerelevee_professionnelle", "consommation_profilee_pro")

    # Consommation Residentielle
    save_path = get_save_path(year, "Consommation_RESIDENTIELLE", "yearly").parent
    plot_time_serie(df, "timestamp", f"Consommation_RESIDENTIELLE-{year}", save_path, "consommation_telerelevee_residentielle", "consommation_profilee_res")
    plot_time_serie_monthly(df, "timestamp", f"Consommation_RESIDENTIELLE-{year}", save_path, "consommation_telerelevee_residentielle", "consommation_profilee_res")

    # Injection
    save_path = get_save_path(year, "Injection", "yearly").parent
    plot_time_serie(df, "timestamp", f"Injection-{year}", save_path, "injection_rte")
    plot_time_serie_monthly(df, "timestamp", f"Injection-{year}", save_path, "injection_rte")

    # Soutirage
    save_path = get_save_path(year, "Soutirage", "yearly").parent
    plot_time_serie(df, "timestamp", f"Soutirage-{year}", save_path, "soutirage_rte", "soutirage_vers_autres_grd")
    plot_time_serie_monthly(df, "timestamp", f"Soutirage-{year}", save_path, "soutirage_rte", "soutirage_vers_autres_grd")

    # Production
    save_path = get_save_path(year, "Production", "yearly").parent
    plot_time_serie(df, "timestamp", f"Production-{year}", save_path, "production_telerelevee", "production_profilee")
    plot_time_serie_monthly(df, "timestamp", f"Production-{year}", save_path, "production_telerelevee", "production_profilee")
    
    save_path = get_save_path(year, "Production-Eolien", "yearly").parent
    plot_time_serie(df, "timestamp", f"Production-Eolien-{year}", save_path, "production_eolien")
    plot_time_serie_monthly(df, "timestamp", f"Production-Eolien-{year}", save_path, "production_eolien")
    
    save_path = get_save_path(year, "Production-Cogeneration", "yearly").parent
    plot_time_serie(df, "timestamp", f"Production-Cogeneration-{year}", save_path, "production_cogeneration", "production_profilee_cogeneration")
    plot_time_serie_monthly(df, "timestamp", f"Production-Cogeneration-{year}", save_path, "production_cogeneration", "production_profilee_cogeneration")
    
    save_path = get_save_path(year, "Production-Photovoltaique", "yearly").parent
    plot_time_serie(df, "timestamp", f"Production-Photovoltaique-{year}", save_path, "production_photovoltaique", "production_profilee_photovoltaique")
    plot_time_serie_monthly(df, "timestamp", f"Production-Photovoltaique-{year}", save_path, "production_photovoltaique", "production_profilee_photovoltaique")
    
    save_path = get_save_path(year, "Production-Autre", "yearly").parent
    plot_time_serie(df, "timestamp", f"Production-Autre-{year}", save_path, "production_autre", "production_profilee_aut")
    plot_time_serie_monthly(df, "timestamp", f"Production-Autre-{year}", save_path, "production_autre", "production_profilee_aut")

    # Temperature
    save_path = get_save_path(year, "Temperature", "yearly").parent
    plot_time_serie(df, "timestamp", f"Temperature-{year}", save_path, "temperature_reelle_lissee", "temperature_normale_lissee")
    plot_time_serie_monthly(df, "timestamp", f"Temperature-{year}", save_path, "temperature_reelle_lissee", "temperature_normale_lissee")
    
    # Ecart Temperature
    save_path = get_save_path(year, "Ecart_Temperature", "yearly").parent
    plot_time_serie(df, "timestamp", f"Ecart_Temperature-{year}", save_path, "ecart_temperature")
    plot_time_serie_monthly(df, "timestamp", f"Ecart_Temperature-{year}", save_path, "ecart_temperature")

    # Degre Jour Chauffage
    save_path = get_save_path(year, "Degre_Jour_Chauffage", "yearly").parent
    plot_time_serie(df, "timestamp", f"Degre_Jour_Chauffage-{year}", save_path, "degre_jour_chauffage")
    plot_time_serie_monthly(df, "timestamp", f"Degre_Jour_Chauffage-{year}", save_path, "degre_jour_chauffage")
    
    # Degre Jour Clim
    save_path = get_save_path(year, "Degre_Jour_Clim", "yearly").parent
    plot_time_serie(df, "timestamp", f"Degre_Jour_Clim-{year}", save_path, "degre_jour_clim")
    plot_time_serie_monthly(df, "timestamp", f"Degre_Jour_Clim-{year}", save_path, "degre_jour_clim")

    # Pseudo Rayonnement
    save_path = get_save_path(year, "Pseudo_Rayonnement", "yearly").parent
    plot_time_serie(df, "timestamp", f"Pseudo Rayonnement-{year}", save_path, "pseudo_rayonnement")
    plot_time_serie_monthly(df, "timestamp", f"Pseudo Rayonnement-{year}", save_path, "pseudo_rayonnement")

    # Price
    save_path = get_save_path(year, "Price", "yearly").parent
    plot_time_serie(df, "timestamp", f"Day-ahead Price (EUR/MWh)-{year}", save_path, "Day-ahead Price (EUR/MWh)")
    plot_time_serie_monthly(df, "timestamp", f"Day-ahead Price (EUR/MWh)-{year}", save_path, "Day-ahead Price (EUR/MWh)")

## **AutoCorrelation and Partial AutoCorrelation Analysis**

We analyze the autocorrelation (ACF) and partial autocorrelation (PACF) for consumption columns to identify seasonal patterns and potential lags for modeling.

In [ ]:
for year in YEARS:
    file_path = PROCESSED_DATA_DIR / f"enedis_dataset_year_{year}.parquet"
    if not file_path.exists():
        continue
        
    print(f"Processing ACF/PACF for year {year}...")
    df = pl.read_parquet(file_path)
    
    consommation_cols = [col for col in df.columns if "consommation" in col and not col.startswith("is_anomaly_")]
    
    for col in consommation_cols:
        save_path = FIGURES_DIR / str(year) / "Consommation" / "ACF_PACF"
        plot_acf_decomposition(df, col, lags=[48, 336], save_path=save_path)
        plot_pacf_decomposition(df, col, lags=[48, 336], save_path=save_path)

Processing ACF/PACF for year 2021...
Processing ACF/PACF for year 2022...
Processing ACF/PACF for year 2023...
Processing ACF/PACF for year 2024...
Processing ACF/PACF for year 2025...
Processing ACF/PACF for year 2026...


## **Stationary Test (ADF)**

We perform the Augmented Dickey-Fuller test on all columns containing 'consommation' to check for stationarity.

In [ ]:
for year in YEARS:
    file_path = PROCESSED_DATA_DIR / f"enedis_dataset_year_{year}.parquet"
    if not file_path.exists():
        continue
        
    print(f"\n{'='*30} Stationary Test for Year {year} {'='*30}")
    df = pl.read_parquet(file_path)
    
    consommation_cols = [col for col in df.columns if "consommation" in col and not col.startswith("is_anomaly_")]
    test_stationarity(df, consommation_cols)

## Execution Loop

In [ ]:
for year in YEARS:
    df = process_year_exploration(year)
    if df is not None:
        df = df.with_columns(
            ecart_temperature=df.get_column("temperature_reelle_lissee") - df.get_column("temperature_normale_lissee"),
            degre_jour_chauffage=(pl.lit(18) - pl.col("temperature_reelle_lissee")).clip(lower_bound=0),
            degre_jour_clim=(pl.col("temperature_reelle_lissee") - pl.lit(21)).clip(lower_bound=0),
        )
        
        # 1. Run Standard Exploration Plots
        run_standard_exploration(df, year)
        
        # 2. Run Anomaly Detection and Labeling
        df = label_and_plot_anomalies(df, year)
        
        # 3. Save the modified dataset with anomaly labels
        output_path = PROCESSED_DATA_DIR / f"enedis_dataset_year_{year}_with_anomalies.parquet"
        df.write_parquet(output_path)
        print(f"Saved dataset with anomaly labels to {output_path}")

## **Time Series Decomposition Analysis**

In [ ]:
from utils.plots import plot_decomposition

column_mapping = {
    "consommation_totale": "Consommation",
    "consommation_telerelevee_hta": "Consommation_HTA",
    "consommation_profilee_ent_hta": "Consommation_HTA",
    "consommation_telerelevee_btsup": "Consommation_BTSUP",
    "consommation_profilee_ent_bt": "Consommation_BTSUP",
    "consommation_telerelevee_professionnelle": "Consommation_PRO",
    "consommation_profilee_pro": "Consommation_PRO",
    "consommation_telerelevee_residentielle": "Consommation_RESIDENTIELLE",
    "consommation_profilee_res": "Consommation_RESIDENTIELLE",
    "production_totale": "Production",
    "production_telerelevee": "Production",
    "production_profilee": "Production",
    "production_eolien": "Production-Eolien",
    "production_cogeneration": "Production-Cogeneration",
    "production_profilee_cogeneration": "Production-Cogeneration",
    "production_photovoltaique": "Production-Photovoltaique",
    "production_profilee_photovoltaique": "Production-Photovoltaique",
    "production_autre": "Production-Autre",
    "production_profilee_aut": "Production-Autre",
    "injection_rte": "Injection",
    "soutirage_rte": "Soutirage",
    "soutirage_vers_autres_grd": "Soutirage",
    "pertes": "Pertes",
    "Day-ahead Price (EUR/MWh)": "Price"
}

periods = [48, 336] # 24h and 1 week

for year in YEARS:
    file_path = PROCESSED_DATA_DIR / f"enedis_dataset_year_{year}.parquet"
    if not file_path.exists():
        continue
        
    print(f"Processing Decomposition for year {year}...")
    df = pl.read_parquet(file_path)
    
    for col, category in column_mapping.items():
        if col not in df.columns:
            continue
            
        save_path = FIGURES_DIR / str(year) / category
        
        for period in periods:
            window_h = period * 0.5
            title = f"Decomposition_{col.replace('/', '_')}_{window_h}h"
            plot_decomposition(df, col, period, save_path, title)